In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/content/drive/MyDrive/REsnet/data.csv")  # <- Your CSV
print(df.head())


                              text     label
0                I love this movie  positive
1                This movie is bad  negative
2          It's okay but not great   neutral
3  Absolutely fantastic experience  positive
4                Worst acting ever  negative


In [3]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
texts = df['text'].values
labels = df['label'].values

In [4]:
x_train, x_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [5]:
vocab_size = 5000
max_len = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

x_train_pad = pad_sequences(x_train_seq, maxlen=max_len, padding='post')
x_test_pad = pad_sequences(x_test_seq, maxlen=max_len, padding='post')


In [6]:
model = models.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_len),
    layers.Conv1D(64, 5, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, 5, activation='relu'),
    layers.GlobalMaxPooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [13]:
model.fit(x_train_pad, y_train, epochs=6, batch_size=32, validation_split=0.2)

Epoch 1/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.3333 - loss: 1.0197 - val_accuracy: 0.0000e+00 - val_loss: 1.1434
Epoch 2/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 1.0000 - loss: 1.0039 - val_accuracy: 0.0000e+00 - val_loss: 1.1445
Epoch 3/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 1.0000 - loss: 0.9863 - val_accuracy: 0.0000e+00 - val_loss: 1.1572
Epoch 4/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 1.0000 - loss: 0.9652 - val_accuracy: 0.0000e+00 - val_loss: 1.1733
Epoch 5/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 1.0000 - loss: 0.9432 - val_accuracy: 0.0000e+00 - val_loss: 1.1870
Epoch 6/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 1.0000 - loss: 0.9191 - val_accuracy: 0.0000e+00 - val_loss: 1.2036


In [14]:
loss, acc = model.evaluate(x_test_pad, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")


✅ Test Accuracy: 0.00%


In [15]:
def predict(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(pad)[0]
    label = le.inverse_transform([pred.argmax()])[0]
    print(f"Sentence: {sentence}")
    print(f"Prediction: {label.upper()}   (Confidence: {pred.max():.2f})\n")

In [16]:
predict("This movie is bad")
predict("It's okay but not great")
predict("Absolutely fantastic experience")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Sentence: This movie is bad
Prediction: NEUTRAL   (Confidence: 0.36)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Sentence: It's okay but not great
Prediction: NEUTRAL   (Confidence: 0.50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Sentence: Absolutely fantastic experience
Prediction: NEUTRAL   (Confidence: 0.36)

